# Pandera Exploration

In [1]:
import pandas as pd
import pandera.pandas as pa             # när man jobbar med pandas DataFrames är det mycket rekommenderat att använda pandera.pandas istället för bara pandera

### DataFrameSchema

In [ ]:
# data att validera
df = pd.DataFrame({
    "station": ["ST1", "ST1", "ST3", "ST5"],
    "kunder": [1000, 400, 84, 2000],
    "avbrottstyp": ["planerat", "oplanerat", "oplanerat", "planerat"]
})

print(df)

In [ ]:
# sätt upp schema
schema = pa.DataFrameSchema({
    "station": pa.Column(
        str,
        pa.Check.isin(["ST1", "ST2", "ST3", "ST4", "ST5"])
    ),
    "kunder": pa.Column(int, pa.Check.ge(0)),
    "avbrottstyp": pa.Column(
        str,
        pa.Check.isin(["planerat", "oplanerat"])
    )
})

In [ ]:
validated_df = schema.validate(df)
print(validated_df)

Då datan stämde returneras df:en utan fel. 

In [ ]:
# data med fel 
df_fel = pd.DataFrame({
    "station": ["ST1", "ST1", "ST3", "ST6"],
    "kunder": [1000, 400, 84, 2000.4],
    "avbrottstyp": ["planerat", "oplanerat", "oplanerat", "planerat"]
})

print(df_fel)

In [ ]:
# validated_df_fel = schema.validate(df_fel)

Koden kraschar med ett SchemaError vid första felet (ST6 är inte en godkänd station)

### Dataframe Model

In [ ]:
# Definiera schema typ som en dataclass
class Schema(pa.DataFrameModel):
    station: str = pa.Field(isin=["ST1", "ST2", "ST3", "ST4", "ST5"])
    kunder: int = pa.Field(ge=0)
    avbrottstyp: str = pa.Field(isin=["planerat", "oplanerat"])

Schema.validate(df)

In [ ]:
# Test med felaktig data
# Schema.validate(df_fel)

Och det blir mycket riktigt ett SchemaError vid första felet.

### Informativa fel

In [ ]:
simple_schema = pa.DataFrameSchema({
    "voltage_level_kv": pa.Column(
        float,
        pa.Check(
            lambda x: 0.0 <= x <= 20.0,
            element_wise=True,
            error="range checker [0, 20]"
        )
    )
})

# datan bryter mot regeln
fail_check_df = pd.DataFrame({
    "voltage_level_kv": [-4.0, 0.4, 10.0, 20.0]
})

try:
    simple_schema(fail_check_df)
except pa.errors.SchemaError as exc:
    print(exc)

In [ ]:
# Med felaktigt kolumnnamn
wrong_column_df = pd.DataFrame({
    "duration_minutes": [5.6, 9.0]
})

try:
    simple_schema(wrong_column_df)
except pa.errors.SchemaError as exc:
    print(exc)

### Felrapporter

In [2]:
# data med fel 
df_fel = pd.DataFrame({
    "station": ["ST1", "ST1", "ST3", "ST6"],
    "kunder": [1000, 400, 84, 2000.4],
    "avbrottstyp": ["planerat", "oplanerat", "oplanerat", "planerat"]
})

print(df_fel)

  station  kunder avbrottstyp
0     ST1  1000.0    planerat
1     ST1   400.0   oplanerat
2     ST3    84.0   oplanerat
3     ST6  2000.4    planerat


In [ ]:
# sätt upp schema
schema = pa.DataFrameSchema({
    "station": pa.Column(
        str,
        pa.Check.isin(["ST1", "ST2", "ST3", "ST4", "ST5"])
    ),
    "kunder": pa.Column(int, pa.Check.ge(0)),
    "avbrottstyp": pa.Column(
        str,
        pa.Check.isin(["planerat", "oplanerat"])
    )
},
name="MySchema",
strict=True                                     # strict=True --> "a column in the dataframe is not specified in the schema", behövs inte för min exempeldata
)

try:
    schema.validate(df_fel, lazy=True)
except pa.errors.SchemaErrors as exc:
    print(exc)

{
    "DATA": {
        "DATAFRAME_CHECK": [
            {
                "schema": "MySchema",
                "column": "station",
                "check": "isin(['ST1', 'ST2', 'ST3', 'ST4', 'ST5'])",
                "error": "Column 'station' failed element-wise validator number 0: isin(['ST1', 'ST2', 'ST3', 'ST4', 'ST5']) failure cases: ST6"
            }
        ]
    },
    "SCHEMA": {
        "WRONG_DATATYPE": [
            {
                "schema": "MySchema",
                "column": "kunder",
                "check": "dtype('int64')",
                "error": "expected series 'kunder' to have type int64, got float64"
            }
        ]
    }
}


"DATAFRAME_CHECK" visar vad det är som är fel (dvs ST6 finns inte med som godkänd station), men WRONG_DATATYPE visar inte vad som är fel (dvs 2000.4 är en float och inte en int). 

Vad beror detta på?

Check.isin(["ST1", "ST2", "ST3", "ST4", "ST5"]) kollar specifikt efter dessa strängar, medan Column(int, pa.Check.ge(0)) bara kollar att hela raden/serien är int. 

Hela kolumnen ser ut att ha blivit float64.

In [6]:
print(df_fel.dtypes)

station            str
kunder         float64
avbrottstyp        str
dtype: object


Just det, Series är ndarray-like (https://pandas.pydata.org/docs/user_guide/dsintro.html#series-is-ndarray-like) och numpy.ndarray är homogen (https://numpy.org/doc/stable/reference/generated/numpy.ndarray.html#numpy.ndarray) vilket innebär att en float gör hela serien till float. 

Hur gör man för att fånga upp detta typ av fel?

### https://pandera.readthedocs.io/en/stable/lazy_validation.html

In [16]:
# Exempel 

df = pd.DataFrame({"kunder": [1000, 400, 84, 2000.4]})

schema = pa.DataFrameSchema({"kunder": pa.Column(int)})

try:
    schema.validate(df)
except pa.errors.SchemaError as exc:
    print(exc)

expected series 'kunder' to have type int64, got float64


https://pandera.readthedocs.io/en/stable/dtype_validation.html#data-type-coercion

-->

https://pandera.readthedocs.io/en/stable/parsers.html 

-->

https://pandera.readthedocs.io/en/stable/dataframe_schemas.html#coerced

Om det smyger sig in 2000.4 i kolumnen "kunder" så kan det ju innebära att hela talet är fel, inte bara att decimalen är fel. Det kanske till exempel skulle vara 20004.

Därför kan det bli dumt att med `coerced=True` göra så att 2000.4 blir en int (dvs 2000). 

Modulusoperatorn (%) ger det som blir över efter division mellan två tal:

In [32]:
print(2000.4 % 1)
print(2000.4 % 2)
print(2000.4 % 3)

0.40000000000009095
0.40000000000009095
2.400000000000091


Med hjälp av modulus går det alltså att kontrollera om decimalen är 0 eller något annat tal:

In [33]:
2000.4 % 1 == 0

False

In [35]:
2000.0 % 1 == 0

True

https://pandera.readthedocs.io/en/stable/checks.html#registering-custom-checks 

--> 

https://pandera.readthedocs.io/en/stable/extensions.html#registering-custom-check-methods 

Men frågan är om det inte är mest korrekt att låta allt krascha om indatan innehåller floats trots att det inte är tillåtet. 

(Och vid krasch köra en separat kontroll för att se vad det är som har blivit fel så det går att fixa)

### Tänkt datamodell

| Kolumnnamn | Dtype | Validering |
| --- | --- | --- |
| incident_id | str | Kolla unique |
| voltage_level_kv | float | Tillåtna värden 0,4, 10.0 och 20.0 |
| start_time | datetime | Giltigt tidsintervall |
| end_time | datetime | Måste vara efter start_time |
| duration_minutes | int | Kontroll mot start_time och end_time |
| outage_type | str | Kontroll mot fasta kategorier | 
| cause_category | str | Kontroll mot fasta kategorier samt att om outage_type == "planerat" så kan inte cause_category vara t ex "väder" |
| customers_affected | int | Heltal, icke-negativt |
| compensation_eligible | bool | Ska vara True om duration_minutes >= 720 minuter och outage_type == "oplanerat" |
